In [ ]:
%python
# ============================================================
# CONFIGURATION — update these two values before running
# ============================================================
catalog = "serverless_stable_ysmqdz_catalog"
schema  = "abac_test"

current_user = spark.sql("SELECT current_user()").collect()[0][0]
print(f"✓ Using: {catalog}.{schema}")
print(f"✓ Running as: {current_user}")


In [ ]:
# ── Cleanup: drop all policies from previous runs to avoid conflicts ──
_all_policies = ['abac_hc_emergency_access', 'abac_hc_regional_access_deny', 'abac_hc_regional_access_east', 'abac_hc_regional_access_north', 'abac_hc_regional_access_south', 'abac_hc_regional_access_west', 'fraud_detection_filter', 'region_row_filter_apac_policy', 'region_row_filter_catalog_apac', 'region_row_filter_catalog_deny', 'region_row_filter_catalog_eu', 'region_row_filter_catalog_us', 'region_row_filter_deny_policy', 'region_row_filter_eu_policy', 'region_row_filter_us_policy', 'flexible_mask_metadata', 'hipaa_mask_contact', 'hipaa_mask_dob', 'hipaa_mask_ids', 'hipaa_mask_name', 'mask_ssn_policy', 'pci_mask_credit_card', 'pci_mask_income', 'pci_mask_ssn', 'pseudo_email_policy', 'redact_other_pii', 'redact_pii', 'clearance_access_policy', 'pii_mask_email', 'pii_mask_name', 'user_access_filter', 'pci_pending_lockdown', 'pci_reviewed_card', 'pci_reviewed_cvv', 'pci_reviewed_name', 'review_complete_classified', 'review_complete_policy', 'review_pending_policy', 'mask_confidential_finance', 'mask_confidential_hr', 'mask_confidential_marketing', 'mask_internal_finance', 'mask_internal_hr', 'mask_internal_marketing', 'telco_billing_confidential', 'telco_billing_restricted', 'telco_cpni_internal', 'telco_cpni_restricted', 'telco_identity_confidential', 'telco_identity_internal', 'telco_identity_restricted', 'telco_service_row_filter_bundle', 'telco_service_row_filter_data', 'telco_service_row_filter_deny', 'telco_service_row_filter_voice']

_dropped, _skipped = 0, 0
for _p in _all_policies:
    try:
        spark.sql(f"DROP POLICY {_p} ON SCHEMA {catalog}.{schema}")
        _dropped += 1
    except Exception:
        _skipped += 1

print(f"Cleanup done — dropped: {_dropped}, not found: {_skipped}")


# Tutorial 5: Multi-Domain Data Governance

This tutorial models a real-world pattern where columns belong to different business domains and have varying sensitivity levels. It combines row filtering (by region) with domain-aware column masking (by sensitivity), all using AND conditions, EXCEPT clauses, and policy composition.

**Pattern:** Each column is tagged with a domain (HR, Finance, Marketing) and a sensitivity level (internal, confidential). Users in a domain group see their domain's columns unmasked. Everyone else sees partial masks (internal) or full redaction (confidential). Rows are filtered by region.

**Requirements:**

- Databricks Runtime 16.4+ or serverless compute
- A Unity Catalog-enabled workspace
- Governed tags created via the Catalog Explorer UI (see Setup section)
- `CREATE SCHEMA`, `CREATE TABLE`, `CREATE FUNCTION` privileges on the target catalog
- Ownership or `MANAGE` privilege on the schema to create policies
- `APPLY TAG` privilege on columns
- `ASSIGN` privilege on the governed tags being used
- Account groups: `abac_tut_hr_team`, `abac_tut_finance_team`, `abac_tut_marketing_team`, `abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_admins`

## The Problem

An organization has an employee database. Different teams need access to different columns:

| Column | Domain | Who should see it |
|--------|--------|-------------------|
| employee_name | HR | HR team |
| ssn | HR | HR team |
| email | Marketing | Marketing team |
| customer_list | Marketing | Marketing team |
| cost_center | Finance | Finance team |
| salary_band | Finance | Finance team |

On top of that, not all sensitive data is equally sensitive:
- **Internal** columns (employee_name, email, cost_center, customer_list) can be partially masked for non-authorized users
- **Confidential** columns (ssn, salary_band) must be fully redacted for non-authorized users

And rows should be filtered by region — US team sees US rows, EU team sees EU rows.

This tutorial shows how to implement all three dimensions (region, domain, sensitivity) with a clean, repeatable policy pattern using AND conditions and EXCEPT clauses.

## Setup

> **Note:** This tutorial uses `{catalog}` as the catalog name. Replace with your own catalog if needed.

### Create account groups

This tutorial uses: `abac_tut_hr_team`, `abac_tut_finance_team`, `abac_tut_marketing_team`, `abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_admins`

Account-level groups must be created in the **Account Console** (Admin > Groups), not via SQL.

> If these groups already exist from previous tutorials, skip this step.

### Create governed tags

Create the following governed tags in the Catalog Explorer UI (**Catalog** > **Governed Tags** > **Create governed tag**):

| Tag Key | Allowed Values |
|---------|---------------|
| `abac_region` | *(none — key-only tag)* |
| `abac_domain` | `hr, finance, marketing` |
| `abac_sensitivity` | `internal, confidential` |

> If `abac_region` or `abac_domain` already exist from previous tutorials, you may only need to create `abac_sensitivity` and add any missing allowed values to `abac_domain`.

In [ ]:
spark.sql(f"""
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


## Step 1: Sample Data

Each sensitive column gets two tags: `abac_domain` (which team owns it) and `abac_sensitivity` (how strictly to mask it). The `region` column gets the `abac_region` tag for row filtering.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.employee_records")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.employee_records (
  id INT,
  employee_name STRING,
  ssn STRING,
  email STRING,
  customer_list STRING,
  cost_center STRING,
  salary_band STRING,
  region STRING,
  department STRING
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.employee_records VALUES
  (1, 'Alice Johnson',  '123-45-6789', 'alice@acme.com',  'Tier-1 Enterprise', 'CC-4010', 'Band 7', 'us', 'Engineering'),
  (2, 'Bob Smith',      '234-56-7890', 'bob@acme.com',    'SMB Accounts',      'CC-3020', 'Band 5', 'us', 'Sales'),
  (3, 'Carol White',    '345-67-8901', 'carol@acme.com',  'Tier-1 Enterprise', 'CC-5010', 'Band 8', 'eu', 'Engineering'),
  (4, 'David Lee',      '456-78-9012', 'david@acme.com',  'Growth Segment',    'CC-2010', 'Band 4', 'eu', 'Marketing'),
  (5, 'Eva Martinez',   '567-89-0123', 'eva@acme.com',    'Mid-Market',        'CC-4020', 'Band 6', 'us', 'HR')
""")


In [ ]:
spark.sql(f"""
-- HR domain: employee_name (internal) + ssn (confidential)
ALTER TABLE {catalog}.{schema}.employee_records
  ALTER COLUMN employee_name SET TAGS ('abac_tut_domain' = 'hr', 'abac_tut_sensitivity' = 'internal')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employee_records
  ALTER COLUMN ssn SET TAGS ('abac_tut_domain' = 'hr', 'abac_tut_sensitivity' = 'confidential')
""")

spark.sql(f"""
-- Marketing domain: email (internal) + customer_list (confidential)
ALTER TABLE {catalog}.{schema}.employee_records
  ALTER COLUMN email SET TAGS ('abac_tut_domain' = 'marketing', 'abac_tut_sensitivity' = 'internal')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employee_records
  ALTER COLUMN customer_list SET TAGS ('abac_tut_domain' = 'marketing', 'abac_tut_sensitivity' = 'confidential')
""")

spark.sql(f"""
-- Finance domain: cost_center (internal) + salary_band (confidential)
ALTER TABLE {catalog}.{schema}.employee_records
  ALTER COLUMN cost_center SET TAGS ('abac_tut_domain' = 'finance', 'abac_tut_sensitivity' = 'internal')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employee_records
  ALTER COLUMN salary_band SET TAGS ('abac_tut_domain' = 'finance', 'abac_tut_sensitivity' = 'confidential')
""")

spark.sql(f"""
-- Region tag for row filtering
ALTER TABLE {catalog}.{schema}.employee_records ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")


## Step 2: UDFs

Three UDFs:
- **Row filter** — checks region group membership
- **Partial mask** — for `internal` columns: first character + `***`
- **Redact** — for `confidential` columns: full replacement

In [ ]:
spark.sql(f"""
-- UDFs: pure data logic, no identity checks
CREATE OR REPLACE FUNCTION {catalog}.{schema}.region_filter_us(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'us'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.region_filter_eu(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN region_val = 'eu'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.region_filter_deny(region_val STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN FALSE
""")

In [ ]:
spark.sql(f"""
-- Partial mask for internal-sensitivity columns
CREATE OR REPLACE FUNCTION {catalog}.{schema}.partial_mask(val STRING)
RETURNS STRING
RETURN CONCAT(LEFT(val, 1), '***')
""")

spark.sql(f"""
-- Full redaction for confidential-sensitivity columns
CREATE OR REPLACE FUNCTION {catalog}.{schema}.redact(val STRING)
RETURNS STRING
RETURN '***REDACTED***'
""")


## Step 3: Row Filter Policy

Filter rows by region. US team sees US employees, EU team sees EU employees. Admins see all rows.

In [ ]:
spark.sql(f"""
-- Policies: group membership defined here, not in UDFs
CREATE OR REPLACE POLICY region_row_filter_us_policy
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.region_filter_us
TO abac_tut_us_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY region_row_filter_eu_policy
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.region_filter_eu
TO abac_tut_eu_team
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

spark.sql(f"""
-- Default deny for users not in any regional group (excluding admins)
CREATE OR REPLACE POLICY region_row_filter_deny_policy
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.region_filter_deny
TO `account users` EXCEPT abac_tut_us_team, abac_tut_eu_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")

In [ ]:
spark.sql(f"""
-- US team sees rows 1, 2, 5 (us region). EU team sees rows 3, 4 (eu region).
SELECT * FROM {catalog}.{schema}.employee_records
""").show(truncate=False)


## Step 4: Domain Column Mask Policies

The key pattern: one policy per domain × sensitivity combination. Each policy uses AND to match columns with a specific domain AND sensitivity level, and EXCEPT to exempt the owning domain group.

```
           | internal (partial_mask) | confidential (redact)  
  HR       | mask_internal_hr        | mask_confidential_hr   
  Finance  | mask_internal_finance    | mask_confidential_finance
  Marketing| mask_internal_marketing  | mask_confidential_marketing
```

Since each column has exactly one domain, each column is matched by exactly one policy per user — no conflicts.

In [ ]:
spark.sql(f"""
-- Internal columns: partial mask for users outside the owning domain
CREATE OR REPLACE POLICY mask_internal_hr
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.partial_mask
TO `account users` EXCEPT abac_tut_hr_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('abac_tut_domain', 'hr')
  AND has_tag_value('abac_tut_sensitivity', 'internal')
) AS m
ON COLUMN m
""")

spark.sql(f"""
CREATE OR REPLACE POLICY mask_internal_finance
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.partial_mask
TO `account users` EXCEPT abac_tut_finance_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('abac_tut_domain', 'finance')
  AND has_tag_value('abac_tut_sensitivity', 'internal')
) AS m
ON COLUMN m
""")

spark.sql(f"""
CREATE OR REPLACE POLICY mask_internal_marketing
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.partial_mask
TO `account users` EXCEPT abac_tut_marketing_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('abac_tut_domain', 'marketing')
  AND has_tag_value('abac_tut_sensitivity', 'internal')
) AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Confidential columns: full redaction for users outside the owning domain
CREATE OR REPLACE POLICY mask_confidential_hr
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.redact
TO `account users` EXCEPT abac_tut_hr_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('abac_tut_domain', 'hr')
  AND has_tag_value('abac_tut_sensitivity', 'confidential')
) AS m
ON COLUMN m
""")

spark.sql(f"""
CREATE OR REPLACE POLICY mask_confidential_finance
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.redact
TO `account users` EXCEPT abac_tut_finance_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('abac_tut_domain', 'finance')
  AND has_tag_value('abac_tut_sensitivity', 'confidential')
) AS m
ON COLUMN m
""")

spark.sql(f"""
CREATE OR REPLACE POLICY mask_confidential_marketing
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.redact
TO `account users` EXCEPT abac_tut_marketing_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('abac_tut_domain', 'marketing')
  AND has_tag_value('abac_tut_sensitivity', 'confidential')
) AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Row filter + column masking now both active.
-- What you see depends on your region groups AND domain groups.
SELECT * FROM {catalog}.{schema}.employee_records
""").show(truncate=False)


### Expected Results

Columns: `id`, `employee_name`, `ssn`, `email`, `customer_list`, `cost_center`, `salary_band`, `region`, `department`

**HR team member in US region** (`abac_tut_hr_team` + `abac_tut_us_team`):

| id | employee_name | ssn | email | customer_list | cost_center | salary_band | region | department |
|---|---|---|---|---|---|---|---|---|
| 1 | Alice Johnson | 123-45-6789 | `a***` | `***REDACTED***` | `C***` | `***REDACTED***` | us | Engineering |
| 2 | Bob Smith | 234-56-7890 | `b***` | `***REDACTED***` | `C***` | `***REDACTED***` | us | Sales |
| 5 | Eva Martinez | 567-89-0123 | `e***` | `***REDACTED***` | `C***` | `***REDACTED***` | us | HR |

HR columns (employee_name, ssn) are unmasked. Marketing internal (email) is partially masked, marketing confidential (customer_list) is redacted. Finance internal (cost_center) is partially masked, finance confidential (salary_band) is redacted.

**Finance team member in EU region** (`abac_tut_finance_team` + `abac_tut_eu_team`):

| id | employee_name | ssn | email | customer_list | cost_center | salary_band | region | department |
|---|---|---|---|---|---|---|---|---|
| 3 | `C***` | `***REDACTED***` | `c***` | `***REDACTED***` | CC-5010 | Band 8 | eu | Engineering |
| 4 | `D***` | `***REDACTED***` | `d***` | `***REDACTED***` | CC-2010 | Band 4 | eu | Marketing |

Finance columns (cost_center, salary_band) are unmasked. All other domains are masked/redacted.

**Admins:** All rows, all columns unmasked.

## Why This Pattern Works

Each column has exactly one `domain` value and one `sensitivity` value, so each column is matched by exactly **one** policy per user. No policy conflicts.

The EXCEPT clause is the key: each policy applies to `account users` EXCEPT the domain group that owns those columns. This means:
- If you're in the owning group → the policy doesn't apply → you see the real data
- If you're NOT in the owning group → the policy applies → data is masked

Multi-group users benefit naturally: a user in both HR and Marketing is excluded from HR policies AND Marketing policies, so they see both domains unmasked.

Adding a new domain (e.g., Legal) requires:
1. A new group (`abac_tut_legal_team`)
2. An allowed value `legal` added to `abac_domain`
3. Two new policies (`mask_internal_legal`, `mask_confidential_legal`)
4. Tag the new columns

No existing policies need to change.

### Delete governed tags

If you no longer need them, delete `abac_sensitivity` from the Catalog Explorer UI (**Governed Tags** section). Note that `abac_region` and `abac_domain` are shared with other tutorials — only delete them if you're done with all tutorials.

---
## Industry Examples

> **Instructions:** Replace `{catalog}` with your Unity Catalog catalog name and `{schema}` with your target schema name before running.
>
> Groups referenced (`abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_apac_team`, `abac_tut_marketing_team`, `abac_tut_finance_team`, `abac_tut_hr_team`, `abac_tut_admins`) must be created in the **Account Console** by an admin.
>
> Governed tags `telco_domain` (with values: cpni, billing, identity) and `telco_sensitivity` (with values: restricted, confidential, internal) must be created in the Catalog Explorer UI.

## Groups Used in Industry Examples

The industry examples below reuse the same account groups from the core tutorial:

| Group | Used As |
|-------|---------|
| `abac_tut_us_team` | Regional (US/North/Voice) |
| `abac_tut_eu_team` | Regional (EU/South/Data) |
| `abac_tut_apac_team` | Regional (APAC/East/Bundle) |
| `abac_tut_admins` | Admin exemption (all policies) |
| `abac_tut_hr_team` | HR domain / Identity owners |
| `abac_tut_finance_team` | Finance domain / Billing / Fraud analysts |
| `abac_tut_marketing_team` | Marketing domain / CPNI owners |

> These groups are managed by your account admin. No group creation SQL is needed here.

## Telco — CPNI Compliance Multi-Domain Governance

Telecommunications providers must protect Customer Proprietary Network Information (CPNI) under FCC regulations (47 U.S.C. § 222). CPNI includes call records, data usage, and account information that reveals service patterns.

Telco data spans three business domains with different sensitivity levels:
- **CPNI domain** (FCC-regulated): call records, data usage — restricted access
- **Billing domain** (financial): monthly spend, account numbers — confidential  
- **Identity domain** (PII): subscriber names, phone numbers, addresses — internal

Each domain team sees their own columns clearly. Other teams see masked or redacted values. Service type teams (voice/data/bundle) further segment row-level access.

**Compliance context:** FCC 47 CFR Part 64 requires telcos to implement safeguards preventing unauthorized disclosure of CPNI. Access must be restricted to personnel with a demonstrated need.

In [ ]:
spark.sql(f"""
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.subscriber_records")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.subscriber_records (
  subscriber_id   INT,
  subscriber_name STRING,
  phone_number    STRING,
  account_number  STRING,
  service_type    STRING,   -- voice, data, bundle
  billing_address STRING,
  data_usage_gb   DOUBLE,
  call_records    STRING,
  plan_type       STRING,
  monthly_spend   DOUBLE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.subscriber_records VALUES
  (1, 'Alice Brown',   '+1-617-555-0101', 'ACCT-001-AB', 'voice',  '123 Oak St, Boston MA',    0.0,   '1200 mins outbound, 800 mins inbound', 'Basic Voice',    45.00),
  (2, 'Bob Davis',     '+1-312-555-0202', 'ACCT-002-BD', 'data',   '456 Elm Ave, Chicago IL',  125.5, 'N/A',                                  'Data Unlimited', 65.00),
  (3, 'Carol Evans',   '+1-512-555-0303', 'ACCT-003-CE', 'bundle', '789 Pine Rd, Austin TX',   88.2,  '650 mins outbound, 420 mins inbound',  'Bundle Pro',     89.00),
  (4, 'David Garcia',  '+1-213-555-0404', 'ACCT-004-DG', 'voice',  '321 Maple Dr, LA CA',      0.0,   '3200 mins outbound, 2100 mins inbound','Premium Voice',  75.00),
  (5, 'Eva Harris',    '+1-206-555-0505', 'ACCT-005-EH', 'data',   '654 Cedar Ln, Seattle WA', 350.0, 'N/A',                                  'Data 5G Plus',   95.00),
  (6, 'Frank Iyer',    '+1-212-555-0606', 'ACCT-006-FI', 'bundle', '987 Birch Blvd, NYC NY',   210.8, '890 mins outbound, 710 mins inbound',  'Bundle Ultimate',120.00),
  (7, 'Grace Johnson', '+1-305-555-0707', 'ACCT-007-GJ', 'voice',  '111 Palm Ave, Miami FL',   0.0,   '450 mins outbound, 380 mins inbound',  'Basic Voice',    45.00),
  (8, 'Henry Kumar',   '+1-415-555-0808', 'ACCT-008-HK', 'data',   '222 Bay St, San Francisco CA', 520.3, 'N/A',                              'Data Enterprise',145.00)
""")


### Tag Columns with Domain and Sensitivity

Each column receives two tags forming a 2D classification grid:
- `telco_domain`: which business domain owns this data
- `telco_sensitivity`: how strictly it must be protected

This enables 6 distinct policies (3 domains × 2 sensitivity levels) to govern all sensitive columns.

In [ ]:
spark.sql(f"""
-- CPNI domain (FCC-regulated)
-- call_records: restricted (most sensitive CPNI)
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN call_records SET TAGS ('telco_domain' = 'cpni', 'telco_sensitivity' = 'restricted')
""")

spark.sql(f"""
-- data_usage_gb: internal (less sensitive CPNI)
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN data_usage_gb SET TAGS ('telco_domain' = 'cpni')
""")

spark.sql(f"""
-- Billing domain (financial data)
-- account_number: restricted
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN account_number SET TAGS ('telco_domain' = 'billing', 'telco_sensitivity' = 'restricted')
""")

spark.sql(f"""
-- monthly_spend: confidential
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN monthly_spend SET TAGS ('telco_domain' = 'billing')
""")

spark.sql(f"""
-- Identity domain (PII)
-- phone_number: restricted (direct identifier)
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN phone_number SET TAGS ('telco_domain' = 'identity', 'telco_sensitivity' = 'restricted')
""")

spark.sql(f"""
-- billing_address: confidential
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN billing_address SET TAGS ('telco_domain' = 'identity', 'telco_sensitivity' = 'confidential')
""")

spark.sql(f"""
-- subscriber_name: internal
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN subscriber_name SET TAGS ('telco_domain' = 'identity', 'telco_sensitivity' = 'internal')
""")




### Row Filter: Service Type Segmentation

Each service team (voice/data/bundle) should only see their customers. Group membership belongs in the **POLICY** (`TO group` / `EXCEPT`), not inside UDF bodies. Each UDF contains only pure data logic.

In [ ]:
spark.sql(f"""
-- UDFs: pure data logic, no identity checks
CREATE OR REPLACE FUNCTION {catalog}.{schema}.service_type_filter_voice(svc_type STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN svc_type = 'voice'
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.service_type_filter_data(svc_type STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN svc_type = 'data'
""")

spark.sql(f"""
-- Bundle team sees bundle, voice, and data to support bundle customers
CREATE OR REPLACE FUNCTION {catalog}.{schema}.service_type_filter_bundle(svc_type STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN svc_type IN ('bundle', 'voice', 'data')
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.service_type_filter_deny(svc_type STRING)
RETURNS BOOLEAN
DETERMINISTIC
RETURN FALSE
""")

In [ ]:
spark.sql(f"""
-- Tag service_type for row filter matching
ALTER TABLE {catalog}.{schema}.subscriber_records ALTER COLUMN service_type SET TAGS ('telco_service_type' = '')
""")

spark.sql(f"""
-- Note: create tag 'telco_service_type' (key-only) in Catalog Explorer first
-- Policies: group membership defined here, not in UDFs
CREATE OR REPLACE POLICY telco_service_row_filter_voice
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.service_type_filter_voice
TO abac_tut_us_team
FOR TABLES
MATCH COLUMNS has_tag('telco_service_type') AS svc_col
USING COLUMNS (svc_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY telco_service_row_filter_data
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.service_type_filter_data
TO abac_tut_eu_team
FOR TABLES
MATCH COLUMNS has_tag('telco_service_type') AS svc_col
USING COLUMNS (svc_col)
""")

spark.sql(f"""
CREATE OR REPLACE POLICY telco_service_row_filter_bundle
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.service_type_filter_bundle
TO abac_tut_apac_team
FOR TABLES
MATCH COLUMNS has_tag('telco_service_type') AS svc_col
USING COLUMNS (svc_col)
""")

spark.sql(f"""
-- Default deny for users not in any service team (excluding admins)
CREATE OR REPLACE POLICY telco_service_row_filter_deny
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.service_type_filter_deny
TO `account users` EXCEPT abac_tut_us_team, abac_tut_eu_team, abac_tut_apac_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('telco_service_type') AS svc_col
USING COLUMNS (svc_col)
""")

### Masking Functions

Three masking functions covering the three sensitivity levels within each domain.

In [ ]:
spark.sql(f"""
-- Phone number masking: ***-***-XXXX
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_phone_number(val STRING)
RETURNS STRING
RETURN CONCAT('***-***-', RIGHT(REPLACE(REPLACE(val, '+1-', ''), '-', ''), 4))
""")


In [ ]:
spark.sql(f"""
-- Account number masking: ACCT-****-XXXX
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_account_number(val STRING)
RETURNS STRING
RETURN CONCAT('ACCT-****-', RIGHT(val, 4))
""")


In [ ]:
spark.sql(f"""
-- Partial redaction: first character + ***
CREATE OR REPLACE FUNCTION {catalog}.{schema}.partial_redact(val STRING)
RETURNS STRING
RETURN CONCAT(LEFT(val, 1), '***')
""")


In [ ]:
spark.sql(f"""
-- Full redaction for restricted data
CREATE OR REPLACE FUNCTION {catalog}.{schema}.full_redact(val STRING)
RETURNS STRING
RETURN '***REDACTED***'
""")

spark.sql(f"""
-- Mask function for DOUBLE columns (must return DOUBLE)
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_double(val DOUBLE)
RETURNS DOUBLE
RETURN 0.0
""")

spark.sql(f"""
-- Apply directly to DOUBLE columns (schema policy can't handle type mismatch)
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN data_usage_gb SET MASK {catalog}.{schema}.mask_double
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.subscriber_records
  ALTER COLUMN monthly_spend SET MASK {catalog}.{schema}.mask_double
""")


### 6 Column Mask Policies: 3 Domains x 2 Sensitivity Levels

Each policy targets a specific domain+sensitivity combination using AND in MATCH COLUMNS. Domain owners are exempted via EXCEPT.

In [ ]:
spark.sql(f"""
-- CPNI Domain Policies
-- Restricted CPNI (call_records): full redact for non-CPNI team
CREATE OR REPLACE POLICY telco_cpni_restricted
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.full_redact
TO `account users` EXCEPT abac_tut_marketing_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('telco_domain', 'cpni')
  AND has_tag_value('telco_sensitivity', 'restricted')
) AS m
ON COLUMN m
""")

spark.sql(f"""
-- Internal CPNI (data_usage_gb): partial redact for non-CPNI team
CREATE OR REPLACE POLICY telco_cpni_internal
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.partial_redact
TO `account users` EXCEPT abac_tut_marketing_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('telco_domain', 'cpni')
  AND has_tag_value('telco_sensitivity', 'internal')
) AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Billing Domain Policies
-- Restricted billing (account_number): account mask format
CREATE OR REPLACE POLICY telco_billing_restricted
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_account_number
TO `account users` EXCEPT abac_tut_finance_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('telco_domain', 'billing')
  AND has_tag_value('telco_sensitivity', 'restricted')
) AS m
ON COLUMN m
""")

spark.sql(f"""
-- Confidential billing (monthly_spend): partial redact
CREATE OR REPLACE POLICY telco_billing_confidential
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.partial_redact
TO `account users` EXCEPT abac_tut_finance_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('telco_domain', 'billing')
  AND has_tag_value('telco_sensitivity', 'confidential')
) AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Identity Domain Policies
-- Restricted identity (phone_number): phone mask format
CREATE OR REPLACE POLICY telco_identity_restricted
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_phone_number
TO `account users` EXCEPT abac_tut_hr_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('telco_domain', 'identity')
  AND has_tag_value('telco_sensitivity', 'restricted')
) AS m
ON COLUMN m
""")

spark.sql(f"""
-- Confidential identity (billing_address): partial redact
CREATE OR REPLACE POLICY telco_identity_confidential
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.partial_redact
TO `account users` EXCEPT abac_tut_hr_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('telco_domain', 'identity')
  AND has_tag_value('telco_sensitivity', 'confidential')
) AS m
ON COLUMN m
""")

spark.sql(f"""
-- Internal identity (subscriber_name, service_type, plan_type): partial redact
CREATE OR REPLACE POLICY telco_identity_internal
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.partial_redact
TO `account users` EXCEPT abac_tut_hr_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS (
  has_tag_value('telco_domain', 'identity')
  AND has_tag_value('telco_sensitivity', 'internal')
) AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Verify: all policies active
-- As abac_tut_us_team member: see only voice subscribers (rows 1, 4, 7)
-- CPNI columns masked unless you're abac_tut_marketing_team
-- Billing columns masked unless you're abac_tut_finance_team  
-- Identity columns partially masked unless you're abac_tut_hr_team
SELECT subscriber_id, subscriber_name, phone_number, account_number,
       service_type, data_usage_gb, call_records, monthly_spend
FROM {catalog}.{schema}.subscriber_records
ORDER BY subscriber_id
""").show(truncate=False)


**Expected results by team membership:**

| Team | Visible Rows | phone_number | account_number | call_records | monthly_spend |
|------|-------------|--------------|----------------|--------------|---------------|
| `abac_tut_us_team` | Voice (1,4,7) | ***-***-XXXX | ACCT-****-XXXX | ***REDACTED*** | 4*** |
| `abac_tut_marketing_team` | By service type | ***-***-XXXX | ACCT-****-XXXX | Full value | 4*** |
| `abac_tut_finance_team` | By service type | ***-***-XXXX | Full value | ***REDACTED*** | Full value |
| `abac_tut_hr_team` | By service type | Full value | ACCT-****-XXXX | ***REDACTED*** | 4*** |
| `abac_tut_admins` | All 8 rows | Full value | Full value | Full value | Full value |